# Experiment 6: Verify and Test API (Postman / Python Requests)

**Objective:**
- Comprehensive API testing including auth, validation & error scenarios
- Test all endpoints systematically
- Use Python requests (equivalent to Postman testing)

**Prerequisites:** Run Experiments 1-5. The API server should be running.

**Postman Installation:**
- Download from https://www.postman.com/downloads/
- Import the collection JSON generated in this notebook

## Step 1: Install Required Libraries

In [ ]:
!pip install requests tabulate colorama

## Step 2: Setup Test Configuration

In [ ]:
import requests
import json
import time
from datetime import datetime

BASE_URL = "http://localhost:8000"
VALID_API_KEY = "mlops-api-key-001"
INVALID_API_KEY = "wrong-key-123"

# Test results tracking
test_results = []

def run_test(test_name, method, url, expected_status, **kwargs):
    """Run a single API test and record result."""
    try:
        start = time.time()
        if method == "GET":
            response = requests.get(url, timeout=10, **kwargs)
        elif method == "POST":
            response = requests.post(url, timeout=10, **kwargs)
        duration = round((time.time() - start) * 1000, 2)
        
        passed = response.status_code == expected_status
        result = {
            "test": test_name,
            "method": method,
            "status": response.status_code,
            "expected": expected_status,
            "passed": "✅ PASS" if passed else "❌ FAIL",
            "duration_ms": duration,
            "response": response.json() if response.headers.get('content-type', '').startswith('application/json') else response.text
        }
    except Exception as e:
        result = {
            "test": test_name,
            "method": method,
            "status": "ERROR",
            "expected": expected_status,
            "passed": "❌ ERROR",
            "duration_ms": 0,
            "response": str(e)
        }
    
    test_results.append(result)
    print(f"{result['passed']} | {test_name} | Status: {result['status']} | {result['duration_ms']}ms")
    return result

print(f"Test configuration ready. Base URL: {BASE_URL}")

## Step 3: Test Public Endpoints

In [ ]:
print("=" * 60)
print("PUBLIC ENDPOINTS")
print("=" * 60)

# Test 1: Root endpoint
r = run_test("GET /", "GET", f"{BASE_URL}/", 200)
print(f"  Response: {r['response']}\n")

# Test 2: Health check
r = run_test("GET /health", "GET", f"{BASE_URL}/health", 200)
print(f"  Response: {r['response']}\n")

# Test 3: API docs
r = run_test("GET /docs", "GET", f"{BASE_URL}/docs", 200)
print(f"  Swagger UI accessible\n")

# Test 4: OpenAPI schema
r = run_test("GET /openapi.json", "GET", f"{BASE_URL}/openapi.json", 200)
if isinstance(r['response'], dict):
    print(f"  API Title: {r['response'].get('info', {}).get('title', 'N/A')}")
    print(f"  API Version: {r['response'].get('info', {}).get('version', 'N/A')}")

## Step 4: Test Authentication

In [ ]:
print("=" * 60)
print("AUTHENTICATION TESTS")
print("=" * 60)

test_payload = {
    "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
    "MonthlyCharges": 70.5, "Contract": "One year",
    "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
}

# Test 5: No auth - should fail
r = run_test("Predict without auth", "POST", f"{BASE_URL}/predict", 401, json=test_payload)
print(f"  Response: {r['response']}\n")

# Test 6: Valid API Key
r = run_test("Predict with valid API Key", "POST", f"{BASE_URL}/predict", 200,
             json=test_payload, headers={"X-API-Key": VALID_API_KEY})
print(f"  Response: {json.dumps(r['response'], indent=2)}\n")

# Test 7: Invalid API Key
r = run_test("Predict with invalid API Key", "POST", f"{BASE_URL}/predict", 403,
             json=test_payload, headers={"X-API-Key": INVALID_API_KEY})
print(f"  Response: {r['response']}\n")

# Test 8: Valid JWT Login
r = run_test("Login with valid credentials", "POST", f"{BASE_URL}/login", 200,
             json={"username": "admin", "password": "admin123"})
jwt_token = r['response'].get('access_token', '')
print(f"  Token received: {jwt_token[:40]}...\n")

# Test 9: Invalid login
r = run_test("Login with invalid credentials", "POST", f"{BASE_URL}/login", 401,
             json={"username": "admin", "password": "wrongpass"})
print(f"  Response: {r['response']}\n")

# Test 10: Predict with JWT
r = run_test("Predict with JWT token", "POST", f"{BASE_URL}/predict", 200,
             json=test_payload, headers={"Authorization": f"Bearer {jwt_token}"})
print(f"  Response: {json.dumps(r['response'], indent=2)}\n")

# Test 11: Predict with expired/invalid JWT
r = run_test("Predict with invalid JWT", "POST", f"{BASE_URL}/predict", 401,
             json=test_payload, headers={"Authorization": "Bearer invalid.token.here"})
print(f"  Response: {r['response']}\n")

## Step 5: Test Input Validation & Error Scenarios

In [ ]:
print("=" * 60)
print("INPUT VALIDATION TESTS")
print("=" * 60)
headers = {"X-API-Key": VALID_API_KEY}

# Test 12: Empty body
r = run_test("Empty request body", "POST", f"{BASE_URL}/predict", 422,
             json={}, headers=headers)
print(f"  Errors: {len(r['response'].get('detail', []))} validation errors\n")

# Test 13: Invalid Gender
invalid = test_payload.copy()
invalid['Gender'] = 'Unknown'
r = run_test("Invalid Gender value", "POST", f"{BASE_URL}/predict", 422,
             json=invalid, headers=headers)
print(f"  Response: {json.dumps(r['response'], indent=2)}\n")

# Test 14: Negative Tenure
invalid = test_payload.copy()
invalid['Tenure'] = -5
r = run_test("Negative Tenure", "POST", f"{BASE_URL}/predict", 422,
             json=invalid, headers=headers)
print(f"  Validation caught negative value\n")

# Test 15: Invalid Contract type
invalid = test_payload.copy()
invalid['Contract'] = 'Three year'
r = run_test("Invalid Contract type", "POST", f"{BASE_URL}/predict", 422,
             json=invalid, headers=headers)
print(f"  Validation caught invalid contract\n")

# Test 16: SeniorCitizen out of range
invalid = test_payload.copy()
invalid['SeniorCitizen'] = 5
r = run_test("SeniorCitizen out of range", "POST", f"{BASE_URL}/predict", 422,
             json=invalid, headers=headers)
print(f"  Validation caught out of range\n")

# Test 17: String instead of number
invalid = test_payload.copy()
invalid['MonthlyCharges'] = 'not_a_number'
r = run_test("String as MonthlyCharges", "POST", f"{BASE_URL}/predict", 422,
             json=invalid, headers=headers)
print(f"  Validation caught type error\n")

# Test 18: Missing required fields
r = run_test("Missing most fields", "POST", f"{BASE_URL}/predict", 422,
             json={"Gender": "Male"}, headers=headers)
print(f"  Errors: {len(r['response'].get('detail', []))} missing fields\n")

# Test 19: Invalid PaymentMethod
invalid = test_payload.copy()
invalid['PaymentMethod'] = 'Cash'
r = run_test("Invalid PaymentMethod", "POST", f"{BASE_URL}/predict", 422,
             json=invalid, headers=headers)
print(f"  Validation caught invalid payment method\n")

## Step 6: Test with Various Customer Profiles

In [ ]:
print("=" * 60)
print("CUSTOMER PROFILE PREDICTIONS")
print("=" * 60)
headers = {"X-API-Key": VALID_API_KEY}

profiles = [
    {"name": "Loyal Long-term Customer", "data": {
        "Gender": "Female", "SeniorCitizen": 0, "Tenure": 60,
        "MonthlyCharges": 50.0, "Contract": "Two year",
        "PaymentMethod": "Bank transfer (automatic)", "TotalCharges": 3000.0
    }},
    {"name": "New High-risk Customer", "data": {
        "Gender": "Male", "SeniorCitizen": 1, "Tenure": 1,
        "MonthlyCharges": 100.0, "Contract": "Month-to-month",
        "PaymentMethod": "Electronic check", "TotalCharges": 100.0
    }},
    {"name": "Mid-tenure Customer", "data": {
        "Gender": "Male", "SeniorCitizen": 0, "Tenure": 24,
        "MonthlyCharges": 75.0, "Contract": "One year",
        "PaymentMethod": "Mailed check", "TotalCharges": 1800.0
    }},
    {"name": "Senior Month-to-month", "data": {
        "Gender": "Female", "SeniorCitizen": 1, "Tenure": 5,
        "MonthlyCharges": 85.0, "Contract": "Month-to-month",
        "PaymentMethod": "Credit card (automatic)", "TotalCharges": 425.0
    }}
]

for profile in profiles:
    response = requests.post(f"{BASE_URL}/predict", json=profile['data'], headers=headers)
    result = response.json()
    print(f"\n📋 {profile['name']}")
    print(f"   Prediction: {result.get('prediction_label', 'N/A')}")
    print(f"   Churn Probability: {result.get('churn_probability', 'N/A')}")
    print(f"   No Churn Probability: {result.get('no_churn_probability', 'N/A')}")

## Step 7: Generate Test Summary

In [ ]:
print("\n" + "=" * 60)
print("TEST SUMMARY")
print("=" * 60)

total = len(test_results)
passed = sum(1 for r in test_results if 'PASS' in r['passed'])
failed = sum(1 for r in test_results if 'FAIL' in r['passed'])
errors = sum(1 for r in test_results if 'ERROR' in r['passed'])

print(f"\nTotal Tests:  {total}")
print(f"Passed:       {passed} ✅")
print(f"Failed:       {failed} ❌")
print(f"Errors:       {errors} ⚠️")
print(f"Pass Rate:    {(passed/total*100):.1f}%")

print(f"\n{'Test':<35} {'Status':<8} {'Expected':<10} {'Result':<10}")
print("-" * 65)
for r in test_results:
    print(f"{r['test']:<35} {str(r['status']):<8} {str(r['expected']):<10} {r['passed']:<10}")

## Step 8: Generate Postman Collection

In [ ]:
# Generate a Postman collection JSON that can be imported
postman_collection = {
    "info": {
        "name": "Bank Churn Prediction API",
        "description": "MLOps - Bank Churn Prediction API Tests",
        "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"
    },
    "variable": [
        {"key": "base_url", "value": "http://localhost:8000"},
        {"key": "api_key", "value": "mlops-api-key-001"},
        {"key": "jwt_token", "value": ""}
    ],
    "item": [
        {
            "name": "Health Check",
            "request": {
                "method": "GET",
                "url": "{{base_url}}/health"
            }
        },
        {
            "name": "Login",
            "request": {
                "method": "POST",
                "url": "{{base_url}}/login",
                "header": [{"key": "Content-Type", "value": "application/json"}],
                "body": {
                    "mode": "raw",
                    "raw": json.dumps({"username": "admin", "password": "admin123"})
                }
            }
        },
        {
            "name": "Predict (API Key)",
            "request": {
                "method": "POST",
                "url": "{{base_url}}/predict",
                "header": [
                    {"key": "Content-Type", "value": "application/json"},
                    {"key": "X-API-Key", "value": "{{api_key}}"}
                ],
                "body": {
                    "mode": "raw",
                    "raw": json.dumps({
                        "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
                        "MonthlyCharges": 70.5, "Contract": "One year",
                        "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
                    })
                }
            }
        },
        {
            "name": "Predict (JWT)",
            "request": {
                "method": "POST",
                "url": "{{base_url}}/predict",
                "header": [
                    {"key": "Content-Type", "value": "application/json"},
                    {"key": "Authorization", "value": "Bearer {{jwt_token}}"}
                ],
                "body": {
                    "mode": "raw",
                    "raw": json.dumps({
                        "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
                        "MonthlyCharges": 70.5, "Contract": "One year",
                        "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
                    })
                }
            }
        },
        {
            "name": "Predict (No Auth - Should Fail)",
            "request": {
                "method": "POST",
                "url": "{{base_url}}/predict",
                "header": [{"key": "Content-Type", "value": "application/json"}],
                "body": {
                    "mode": "raw",
                    "raw": json.dumps({
                        "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
                        "MonthlyCharges": 70.5, "Contract": "One year",
                        "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
                    })
                }
            }
        }
    ]
}

with open('postman_collection.json', 'w') as f:
    json.dump(postman_collection, f, indent=2)

print("Postman collection exported to postman_collection.json")
print("Import this file in Postman to test the API interactively.")
print("\n✅ All API tests completed!")